# RepliTaliNorm

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
4. [Load features](#Load-features)
5. [Load weights into base model](#Load-weights-into-base-model)
6. [Load reference values](#Load-reference-values)
7. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
8. [Check all clock parameters](#Check-all-clock-parameters)
9. [Normal feature ranges](#Normal-feature-ranges)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.RepliTaliNorm)

class RepliTaliNorm(LinearReferenceClock):
    pass



In [3]:
model = pya.models.RepliTaliNorm()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "replitalinorm"
model.metadata["data_type"] = "DNA methylation"  # Paper: Both models use DNA methylation at CpGs in common partially methylated domains.
model.metadata["species"] = "Homo sapiens"  # Paper: The models were developed in primary human cells.
model.metadata["year"] = 2022
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Endicott, J.L., Nolte, P.A., Shen, H. & Laird, P.W. Cell division drives DNA methylation loss in late-replicating domains in primary human cells. Nature Communications 13, 6659 (2022)."
model.metadata["doi"] = "https://doi.org/10.1038/s41467-022-34268-8"
model.metadata["notes"] = "Upstream starting-PD normalization model used during RepliTali construction. It was trained only in the chronologically youngest fetal skin fibroblast line (AG06561) to estimate the unobserved pre-culture replicative-history offset; it is not the final 87-CpG RepliTali model."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["cultured fibroblasts"]  # Paper: The normalizer was trained on the chronologically youngest primary cell, fetal skin fibroblast AG06561.
model.metadata["predicts"] = ["replicative history"]  # Paper: The model estimates the starting-passage PD used to correct unknown pre-culture cell divisions.
model.metadata["training_target"] = ["population doublings"]  # Paper: The normalizer was fitted to the measured PD trajectory of AG06561.
model.metadata["unit"] = ["population doublings"]  # Paper: The calibration outcome is cumulative population doublings.
model.metadata["model_type"] = "elastic net regression"  # Paper: Both the starting-PD normalizer and final RepliTali use elastic-net regression with alpha 0.5.
model.metadata["platform"] = ["Illumina EPIC"]  # Paper: Training methylation was measured using the Infinium MethylationEPIC array.
model.metadata["population"] = "human cell cultures"  # Paper: Only the chronologically youngest fetal fibroblast line was used to train the starting-PD normalizer.
model.metadata["journal"] = "Nature Communications"
model.metadata["last_author"] = "Peter W. Laird"
model.metadata["n_features"] = 218
model.metadata["citations"] = 86
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

In [5]:
os.system(f"curl -sL -o RepliTaliNorm_CpGs.rda https://raw.githubusercontent.com/HigginsChenLab/methylCIPHER/19b12296b0d7eb7055a97d068064df635f44ce3e/data/RepliTaliNorm_CpGs.rda")
os.system(f"curl -sL -o RepliTaliNorm_ref.rda https://raw.githubusercontent.com/HigginsChenLab/methylCIPHER/19b12296b0d7eb7055a97d068064df635f44ce3e/data/RepliTaliNorm_ref.rda")

0

In [6]:
%%writefile download.r

library(jsonlite)
load("RepliTaliNorm_CpGs.rda")
load("RepliTaliNorm_ref.rda")
write_json(RepliTaliNorm_CpGs, "coefficients.json", digits = 10)
write_json(RepliTaliNorm_ref$intercept, "intercept.json", digits = 10)

Writing download.r


In [7]:
os.system("Rscript download.r")

0

## Load features

In [8]:
coef_df = pd.DataFrame(json.load(open('coefficients.json')))
model.features = coef_df['CpG'].tolist()
iv = json.load(open('intercept.json'))
intercept_value = iv[0] if isinstance(iv, list) else iv

## Load weights into base model

In [9]:
weights = torch.tensor(coef_df['coefficient'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([intercept_value]).float()

In [10]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [11]:
model.reference_values = None

## Load preprocess and postprocess objects

In [12]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [13]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [14]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Endicott, Jamie L., et al. "Cell division drives DNA methylation '
             'loss in late-replicating domains in primary human cells." Nature '
             'Communications 13.1 (2022): 6659.',
 'clock_name': 'replitalinorm',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1038/s41467-022-34268-8',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2022}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg00065944', 'cg00077044', 'cg00156723', 'cg00278655', 'cg00434151', 'cg00564996', 'cg00824087', 'cg00832327', 'cg01052456', 'cg01129509', 'cg01249202', 'cg01463644', 'cg01922998', 'cg02002355', 'cg02110836', 'cg02148017', 'cg02576202', 'cg02607176', 'cg02653473', 'cg

## Normal feature ranges

In [ ]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

## Basic test

In [ ]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

## Save torch model

In [16]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [17]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: intercept.json
Deleted file: download.r
Deleted file: RepliTaliNorm_CpGs.rda
Deleted file: RepliTaliNorm_ref.rda
Deleted file: coefficients.json
